# 07 — S5P CDSE plan (hardened)
Creates a non-fatal planning manifest if CDSE credentials are missing.

In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import yaml

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def safe_json_response(resp):
    try:
        return resp.json(), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

def env(name):
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

ROOT = find_repo_root(Path.cwd())
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
cfg = yaml.safe_load((CONFIGS / "run.yml").read_text(encoding="utf-8")) if (CONFIGS / "run.yml").exists() else {}
sites = cfg.get("sites", [])
run_cfg = cfg.get("run", {})
DATE_FROM = run_cfg.get("date_from", "2025-01-01")
DATE_TO = run_cfg.get("date_to", "2025-01-31")

def manifest_base(step):
    return {
        "step": step,
        "created_at_utc": utc_now(),
        "repo_root": str(ROOT),
        "configs": [{"path": str(CONFIGS / "run.yml"), "sha256": sha256_file(CONFIGS / "run.yml")}] if (CONFIGS / "run.yml").exists() else [],
        "artifacts": [],
        "notes": [],
        "status": "ok",
    }

def add_artifact(mf, p: Path, meta=None):
    if p.exists():
        row = {"path": str(p), "sha256": sha256_file(p)}
        if meta:
            row["meta"] = meta
        mf["artifacts"].append(row)

In [ ]:
step = "07_s5p_cdse_plan"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

user = env("CDSE_USERNAME") or env("CDSE_USER")
pwd = env("CDSE_PASSWORD")
mf = manifest_base(step)

plan_rows = []
for s in sites:
    plan_rows.append({
        "site_id": s.get("id"),
        "site_name": s.get("name"),
        "lat": s.get("lat"),
        "lon": s.get("lon"),
        "date_from": DATE_FROM,
        "date_to": DATE_TO,
        "cdse_ready": bool(user and pwd),
    })
df = pd.DataFrame(plan_rows)
df.to_csv(out / "s5p_plan.csv", index=False)
add_artifact(mf, out / "s5p_plan.csv", {"rows": int(len(df))})

if not (user and pwd):
    mf["status"] = "skipped_missing_credentials"
    mf["notes"].append("Missing CDSE credentials; planning only.")
write_json(out / "manifest.json", mf)
display(df)